# MODULE 6 — Full LOSO Evaluation (All 9 Subjects)
**Requires:** `eeg_data_bundle.pkl` from Module 1  
**Prerequisite:** Modules 3–5 must all pass first.

**Goal:** Reproduce paper Table 3 & 4 results.

| Config | Paper result |
|--------|--------------|
| CRNN-DF (no GAN) | 63.52 ± 10.70% |
| CRNN-DF + FBGAN  | 72.74 ± 10.44% |

> `USE_GAN = False` → ~30–50 min.  `USE_GAN = True` → ~4–6 hrs.

In [1]:
import numpy as np
import scipy.signal as sig
import matplotlib.pyplot as plt
import pickle, warnings, time
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
IS_MPS = (device.type == 'mps')
print(f'Device: {device}')

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

with open('eeg_data_bundle.pkl', 'rb') as f:
    d = pickle.load(f)
X_raw    = d['X_raw'];    y_enc    = d['y_enc']
subjects = d['subjects']; sessions = d['sessions']
S0=d['S0']; S1=d['S1'];   CLASSES  = d['classes']
FS = d['FS']
print(f'Loaded. X:{X_raw.shape}  subjects:{np.unique(subjects)}')


Device: mps
Loaded. X:(5184, 22, 1000)  subjects:['1' '2' '3' '4' '5' '6' '7' '8' '9']


In [2]:
# ════════════════════════════════════════════════════════════════════════
# All model components — copied clean from Modules 3–5
# ════════════════════════════════════════════════════════════════════════

# ─── Preprocessing ────────────────────────────────────────────────────────
def zscore(X_tr, X_other):
    mu  = X_tr.mean(axis=(0,2), keepdims=True)
    std = X_tr.std (axis=(0,2), keepdims=True) + 1e-8
    return (X_tr - mu) / std, (X_other - mu) / std

def make_loader(X, y, bs=32, shuffle=True):
    return DataLoader(
        TensorDataset(torch.FloatTensor(X), torch.LongTensor(y)),
        batch_size=bs, shuffle=shuffle, drop_last=True)

def predict(model, X, bs=64):
    model.eval(); preds = []
    Xt = torch.FloatTensor(X)
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            l, _ = model(Xt[i:i+bs].to(device))
            preds.extend(l.argmax(1).cpu().numpy())
    return np.array(preds)

# ─── CRNN ─────────────────────────────────────────────────────────────────
class CRNN(nn.Module):
    def __init__(self, n_ch=22, n_t=1000, n_cls=4, hidden=64, dropout=0.5):
        super().__init__()
        self.feature_dim  = hidden
        self.spatial_conv = nn.Conv2d(1, 32, kernel_size=(n_ch, 45), padding=0)
        self.bn           = nn.BatchNorm2d(32)
        self.relu         = nn.ReLU(inplace=True)
        self.drop_cnn     = nn.Dropout(dropout)
        self.pool         = nn.MaxPool2d((1, 75), stride=(1, 10))
        lstm_drop = 0.0 if IS_MPS else dropout
        self.lstm         = nn.LSTM(32, hidden, num_layers=2,
                                    batch_first=True, dropout=lstm_drop)
        self.drop_lstm    = nn.Dropout(dropout)
        self.classifier   = nn.Linear(hidden, n_cls)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.drop_cnn(self.relu(self.bn(self.spatial_conv(x))))
        x = self.pool(x)
        x = x.squeeze(2).permute(0, 2, 1)
        _, (h, _) = self.lstm(x)
        feat = self.drop_lstm(h[-1])
        return self.classifier(feat), feat

# ─── CenterLoss ───────────────────────────────────────────────────────────
class CenterLoss(nn.Module):
    def __init__(self, n_cls=4, feat_dim=64, lr_c=0.5):
        super().__init__()
        self.n_cls = n_cls; self.lr_c = lr_c
        self.register_buffer('centers', torch.zeros(n_cls, feat_dim))

    @torch.no_grad()
    def init_from_data(self, model, X, y, bs=64):
        model.eval()
        all_f, all_l = [], []
        Xt = torch.FloatTensor(X); yt = torch.LongTensor(y)
        for i in range(0, len(Xt), bs):
            _, f = model(Xt[i:i+bs].to(device))
            all_f.append(f.detach().cpu()); all_l.append(yt[i:i+bs])
        F = torch.cat(all_f); L = torch.cat(all_l)
        for c in range(self.n_cls):
            m = (L == c)
            if m.sum() > 0:
                self.centers[c] = F[m].mean(0).to(self.centers.device)
        model.train()

    @torch.no_grad()
    def update_centers(self, feats, labels):
        feats = feats.detach(); labels = labels.detach()
        for c in range(self.n_cls):
            mask = (labels == c)
            if mask.sum() == 0: continue
            diff = self.centers[c] - feats[mask].mean(0)
            self.centers[c] -= self.lr_c * diff / (mask.sum().float() + 1)

    @torch.no_grad()
    def expand_inter_class(self, alpha=0.02):
        gc = self.centers.mean(0, keepdim=True)
        d  = self.centers - gc
        self.centers.add_(alpha * d / (d.norm(dim=1, keepdim=True) + 1e-8))

    def forward(self, feats, labels):
        c = self.centers[labels].detach()
        return torch.mean(torch.norm(feats - c, dim=1))

# ─── FBGAN ────────────────────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, noise_dim=1600, n_ch=22, n_t=1000):
        super().__init__()
        self.n_ch = n_ch; self.n_t = n_t
        self.fc  = nn.Linear(noise_dim, 128 * 4 * 4)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128,128,(3,15),(1,3),(1,6)), nn.BatchNorm2d(128), nn.LeakyReLU(0.2,True),
            nn.ConvTranspose2d(128,128,(3,15),(1,3),(1,6)), nn.BatchNorm2d(128), nn.LeakyReLU(0.2,True),
            nn.ConvTranspose2d(128, 64,(3, 5),(1,2),(1,2)), nn.BatchNorm2d(64),  nn.LeakyReLU(0.2,True),
            nn.ConvTranspose2d( 64, 32,(4, 5),(2,1),(1,2)), nn.BatchNorm2d(32),  nn.LeakyReLU(0.2,True),
            nn.ConvTranspose2d( 32,  1,(1, 2),(1,1),(0,0)), nn.Tanh())
    def forward(self, z):
        x = self.fc(z).view(-1, 128, 4, 4)
        return nn.functional.interpolate(
            self.net(x), size=(self.n_ch, self.n_t),
            mode='bilinear', align_corners=False)

class DiscriminatorPhi(nn.Module):
    def __init__(self, n_ch=22, n_t=1000):
        super().__init__()
        self.act = nn.LeakyReLU(0.2, True)
        self.c1  = nn.Conv2d( 1, 10, (1,23), (1,1), (0,11))
        self.c2  = nn.Conv2d(10, 30, (n_ch,1))
        self.c3  = nn.Conv2d(30, 30, (1,17), (1,1), (0,8))
        self.p1  = nn.MaxPool2d((1,6), (1,6))
        self.c4  = nn.Conv2d(30, 30, (1,7),  (1,7))
        self.p2  = nn.MaxPool2d((1,6), (1,6))
        t = n_t
        for k,s,p in [(23,1,11),(17,1,8),(6,6,0),(7,7,0),(6,6,0)]:
            t = max(1, (t+2*p-k)//s+1)
        self.flat = 30*max(t,1); self.fc = nn.Linear(self.flat, 1)
    def forward(self, x):
        x = self.act(self.c1(x)); x = self.act(self.c2(x))
        x = self.act(self.c3(x)); x = self.p1(x)
        x = self.act(self.c4(x)); x = self.p2(x)
        x = x.reshape(x.size(0), -1)
        if x.shape[1] != self.flat:
            buf = x.new_zeros(x.size(0), self.flat)
            buf[:, :min(x.shape[1],self.flat)] = x[:, :min(x.shape[1],self.flat)]
            x = buf
        return torch.sigmoid(self.fc(x))

class DiscriminatorPsi(nn.Module):
    def __init__(self, n_sparse=20):
        super().__init__()
        self.act = nn.LeakyReLU(0.2, True)
        k1 = min(23, max(3, n_sparse)); p1 = k1//2
        self.c1 = nn.Conv2d( 1, 10, (1,k1), (1,1), (0,p1))
        self.c2 = nn.Conv2d(10, 30, (4, 1), (4,1))
        self.c3 = nn.Conv2d(30, 30, (1, 1))
        self.c4 = nn.Conv2d(30, 30, (1,17), (1,1), (0,8))
        self.p1 = nn.MaxPool2d((1,6), (1,6))
        self.c5 = nn.Conv2d(30, 30, (1, 7))
        self.p2 = nn.MaxPool2d((1,6), (1,6))
        self.fc = nn.Linear(30, 1)
    def forward(self, x):
        x = self.act(self.c1(x)); x = self.act(self.c2(x))
        x = self.act(self.c3(x)); x = self.act(self.c4(x))
        if x.shape[3] >= 6: x = self.p1(x)
        if x.shape[3] >= 7: x = self.act(self.c5(x))
        if x.shape[3] >= 6: x = self.p2(x)
        x = x.mean(dim=[2, 3])   # global avg pool — always safe
        return torch.sigmoid(self.fc(x))

print('All components defined. ✅')


All components defined. ✅


In [3]:
def train_crnn_df(X_tr, y_tr, n_epochs=150, lam=0.05, warmup=50):
    model = CRNN().to(device)
    closs = CenterLoss().to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-6)
    ce    = nn.CrossEntropyLoss()
    loader= make_loader(X_tr, y_tr)
    for ep in range(1, n_epochs + 1):
        if ep == warmup + 1:
            closs.init_from_data(model, X_tr, y_tr)
        use_c = (ep > warmup)
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            logits, feats = model(Xb)
            loss = ce(logits, yb)
            if use_c:
                loss = loss + lam * closs(feats, yb)
                closs.update_centers(feats, yb)
            loss.backward(); opt.step()
        sched.step()
        # inter-class expansion disabled
        # if use_c and ep % 15 == 0: closs.expand_inter_class()
    return model


def run_fbgan(X_calib, y_calib, n_sparse, n_epochs=150, n_fake=750, n_cls=4):
    n_ch, n_t = X_calib.shape[1], X_calib.shape[2]
    bce = nn.BCELoss(); sl = slice(0, min(n_sparse, n_t))
    all_fakes = []
    for cls in range(n_cls):
        X_cls = X_calib[y_calib == cls]
        if len(X_cls) < 8:
            X_cls = np.tile(X_cls, (int(np.ceil(8/max(len(X_cls),1))),1,1))[:8]
        loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_cls[:, np.newaxis])),
            batch_size=8, shuffle=True, drop_last=True)
        G   = Generator(1600, n_ch, n_t).to(device)
        Dp  = DiscriminatorPhi(n_ch, n_t).to(device)
        Ds  = DiscriminatorPsi(n_sparse).to(device)
        o_G = optim.Adam(G.parameters(), lr=1e-4, betas=(0.5,0.999))
        o_D = optim.Adam(list(Dp.parameters())+list(Ds.parameters()),
                         lr=1e-4, betas=(0.5,0.999))
        for _ in range(n_epochs):
            for (real,) in loader:
                real = real.to(device); bs = real.size(0)
                ones  = torch.ones(bs,1,device=device)
                zeros = torch.zeros(bs,1,device=device)
                o_D.zero_grad()
                with torch.no_grad():
                    fake = G(torch.randn(bs,1600,device=device))
                (bce(Dp(real),ones)+bce(Dp(fake),zeros)+
                 bce(Ds(real[:,:,:,sl]),ones)+bce(Ds(fake[:,:,:,sl]),zeros)).backward()
                o_D.step()
                o_G.zero_grad()
                fake = G(torch.randn(bs,1600,device=device))
                (bce(Dp(fake),ones)+bce(Ds(fake[:,:,:,sl]),ones)).backward()
                o_G.step()
        G.eval(); chunks=[]
        with torch.no_grad():
            for i in range(0, n_fake, 32):
                z = torch.randn(min(32,n_fake-i),1600,device=device)
                chunks.append(G(z).squeeze(1).cpu().numpy())
        all_fakes.append(np.vstack(chunks)[:n_fake])
    fake_X = np.vstack(all_fakes).astype(np.float32)
    fake_y = np.repeat(np.arange(n_cls), n_fake).astype(np.int64)
    return fake_X, fake_y

print('Training functions defined. ✅')


Training functions defined. ✅


In [4]:
# ══════════════════════════════════════════════════
#   CONFIGURATION — edit here only
# ══════════════════════════════════════════════════
USE_GAN     = False  # True → full paper result, ~4-6 hrs
                     # False → CRNN-DF only,     ~30-50 min
N_FAKE      = 750    # fake trials per class (paper value)
GAN_EPOCHS  = 150    # GAN epochs per class
CRNN_EPOCHS = 150    # classifier epochs
LAM         = 0.05    # center loss weight λ
WARMUP      = 50     # CE-only warmup epochs before center loss
# ══════════════════════════════════════════════════
print(f'USE_GAN={USE_GAN}  N_FAKE={N_FAKE}  '
      f'CRNN_EPOCHS={CRNN_EPOCHS}  λ={LAM}  warmup={WARMUP}')


USE_GAN=False  N_FAKE=750  CRNN_EPOCHS=150  λ=0.05  warmup=50


In [ ]:
results     = {}   # subj → accuracy
all_preds   = {}   # subj → pred array  (for confusion matrices)
all_true    = {}   # subj → true labels
all_subjs   = sorted(np.unique(subjects))
t_total     = time.time()

for subj in all_subjs:
    t_s = time.time()
    print(f'\n══════ LOSO  test={subj} ══════')

    m_tr  = subjects != subj
    m_cal = (subjects == subj) & (sessions == S0)
    m_te  = (subjects == subj) & (sessions == S1)

    X_tr_n,  X_te_n = zscore(X_raw[m_tr], X_raw[m_te])
    _,       X_ca_n = zscore(X_raw[m_tr], X_raw[m_cal])
    y_tr = y_enc[m_tr]; y_ca = y_enc[m_cal]; y_te = y_enc[m_te]

    print(f'  train={X_tr_n.shape[0]}  calib={X_ca_n.shape[0]}  test={X_te_n.shape[0]}')

    if USE_GAN:
        print('  Running FBGAN...')
        fake_X, fake_y = run_fbgan(
            X_ca_n, y_ca, n_sparse=20,
            n_epochs=GAN_EPOCHS, n_fake=N_FAKE)
        X_aug = np.vstack([X_tr_n, fake_X])
        y_aug = np.hstack([y_tr,   fake_y])
        print(f'  Augmented: {X_aug.shape[0]} trials')
    else:
        X_aug, y_aug = X_tr_n, y_tr

    print('  Training CRNN-DF...')
    model = train_crnn_df(X_aug, y_aug,
                          n_epochs=CRNN_EPOCHS, lam=LAM, warmup=WARMUP)


    preds = predict(model, X_te_n)
    acc   = accuracy_score(y_te, preds) * 100
    results[subj]   = acc
    all_preds[subj] = preds
    all_true[subj]  = y_te

    elapsed = time.time() - t_s
    print(f'  ✅  subj {subj}: {acc:.2f}%   ({elapsed/60:.1f} min)')

total_min = (time.time() - t_total) / 60
vals = list(results.values())
mean_acc, std_acc = np.mean(vals), np.std(vals)

print('\n' + '═'*50)
print('FINAL RESULTS')
print('═'*50)
for s, a in results.items():
    print(f'  Subject {s}: {a:.2f}%')
print(f'  ──────────────────────')
print(f'  Mean ± Std : {mean_acc:.2f} ± {std_acc:.2f}%')
print(f'  Total time : {total_min:.1f} min')
print('─'*50)
print(f'  Paper CRNN-DF  : 63.52 ± 10.70%')
print(f'  Paper + FBGAN  : 72.74 ± 10.44%')
print('═'*50)



══════ LOSO  test=1 ══════
  train=4608  calib=288  test=288
  Training CRNN-DF...


In [ ]:
# ─── Per-subject accuracy bar chart ──────────────────────────────────────
paper_per_subj = {
    '1':65.51,'2':45.18,'3':78.62,'4':53.58,
    '5':55.64,'6':56.03,'7':71.28,'8':75.02,'9':70.78
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

subj_list  = sorted(results.keys())
our_vals   = [results[s]              for s in subj_list]
paper_vals = [paper_per_subj.get(s,0) for s in subj_list]
x = np.arange(len(subj_list))

axes[0].bar(x-0.2, paper_vals, 0.35, label='Paper CRNN-DF', color='#90CAF9', edgecolor='k')
axes[0].bar(x+0.2, our_vals,   0.35, label='Ours',          color='#FF7043', edgecolor='k')
axes[0].axhline(25,     color='gray',   ls=':',  lw=1)
axes[0].axhline(mean_acc, color='crimson', ls='--', lw=1.5,
                label=f'Our mean {mean_acc:.1f}%')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'A{s}' for s in subj_list])
axes[0].set_ylabel('Accuracy (%)'); axes[0].set_ylim(0, 100)
axes[0].set_title('Per-subject Accuracy: Paper vs Ours', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)
for i, (p, o) in enumerate(zip(paper_vals, our_vals)):
    axes[0].text(i-0.2, p+1, f'{p:.0f}', ha='center', fontsize=7, color='navy')
    axes[0].text(i+0.2, o+1, f'{o:.0f}', ha='center', fontsize=7, color='darkred')

# ─── Method comparison bar ────────────────────────────────────────────────
baselines = {
    'EEGNet':51.32, 'CTCNN':47.67, 'AE+\nXGBoost':33.18,
    'FBCSP':35.69,  'CRAM':59.22,
    'Paper\nCRNN-DF':63.52, 'Paper\n+GAN':72.74,
    'Ours':mean_acc
}
bc = ['#B0BEC5']*5 + ['#42A5F5','#1565C0','#FF7043']
axes[1].bar(range(len(baselines)), list(baselines.values()),
            color=bc, edgecolor='k', linewidth=0.7)
axes[1].set_xticks(range(len(baselines)))
axes[1].set_xticklabels(list(baselines.keys()), fontsize=9)
for i, v in enumerate(baselines.values()):
    axes[1].text(i, v+0.5, f'{v:.1f}', ha='center', fontsize=8)
axes[1].axhline(25, color='gray', ls=':', lw=1)
axes[1].set_ylim(0, 90); axes[1].set_ylabel('Mean Accuracy (%)')
axes[1].set_title('Comparison with Published Baselines', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

config_str = 'CRNN-DF + FBGAN' if USE_GAN else 'CRNN-DF (no GAN)'
plt.suptitle(f'Full LOSO Results — {config_str}\n'
             f'Our result: {mean_acc:.2f} ± {std_acc:.2f}%',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('m6_results_bars.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
axes_flat = axes.flatten()

for idx, subj in enumerate(subj_list):
    cm = confusion_matrix(all_true[subj], all_preds[subj])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_flat[idx],
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.5, cbar=False)
    axes_flat[idx].set_title(
        f'Subject {subj}  ({results[subj]:.1f}%)', fontweight='bold', fontsize=10)
    axes_flat[idx].set_xlabel('Predicted', fontsize=8)
    axes_flat[idx].set_ylabel('True', fontsize=8)
    axes_flat[idx].tick_params(labelsize=8)

plt.suptitle(
    f'Confusion Matrices — All 9 Subjects\n'
    f'Mean accuracy: {mean_acc:.2f} ± {std_acc:.2f}%',
    fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('m6_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
methods = ['Paper\nCRNN-DF', 'Paper\n+GAN', 'Ours']
means   = [63.52,  72.74,  mean_acc]
stds    = [10.70,  10.44,  std_acc]
colors  = ['#42A5F5', '#1565C0', '#FF7043']

bars = ax.bar(methods, means, yerr=stds, color=colors,
              edgecolor='k', capsize=10, width=0.5, error_kw={'lw':2})
ax.axhline(25, color='gray', ls=':', lw=1.5, label='Chance (25%)')
ax.set_ylim(0, 95); ax.set_ylabel('Mean ± Std Accuracy (%)', fontsize=12)
ax.set_title('Summary vs Paper Benchmarks', fontweight='bold', fontsize=13)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 1.5, f'{m:.1f}±{s:.1f}%',
            ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('m6_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅  MODULE 6 COMPLETE')
print(f'    {"CRNN-DF+FBGAN" if USE_GAN else "CRNN-DF"}: {mean_acc:.2f} ± {std_acc:.2f}%')
